# 03 — Profiling & Silver Cleaning.


O Data Profiling vai ajudar-nos a entender a qualidade dos dados (nulos, duplicados e distribuição). A Silver Cleaning aplicará as regras de negócio para corrigir esses problemas e ajustar os esquemas (schema casting).


### 3.1 Setup inicial
Nesta fase vamos realizar os imports, carregar as tabelas previamente guardadas em formato Delta e definir os caminhos base de cada uma delas.


In [0]:
# Setup inicial
# imports necessários, definição de caminhos base necessários e lista com as tabelas a trabalhar
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Caminhos Delta necessários para correr este notebook de forma independente
bronze_delta_path = "/Volumes/main/default/faers_data/delta/bronze"
silver_delta_path = "/Volumes/main/default/faers_data/delta/silver"

tables = ["demo", "drug", "reac", "outc"]





In [0]:
# definição de função para analise de nulos

def analisar_nulos(dfs_dict, return_snapshots=True):
    """
    Analisa valores nulos em múltiplas tabelas de forma eficiente.
    
    Parâmetros:
    -----------
    dfs_dict : dict
        Dicionário com nome_tabela -> DataFrame
    return_snapshots : bool
        Se True, retorna dicionário com métricas detalhadas
    
    Retorna:
    --------
    dict : Dicionário com métricas de nulos por tabela (se return_snapshots=True)
    """
    
    snapshots = {}
    
    for table_name, df in dfs_dict.items():
        print(f"--- Tabela: {table_name.upper()} ---")
        
        # Cache das colunas para evitar múltiplas chamadas Analyze RPC
        cols = df.columns
        total_rows = df.count()
        
        # Contagem eficiente: uma única passagem pelos dados
        null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in cols]
        null_counts_row = df.select(*null_exprs).collect()[0]
        
        # Calcular total de valores nulos
        total_nulls = sum([null_counts_row[c] for c in cols])
        
        # Mostrar apenas o total
        print(f"  Total de registos: {total_rows:,}")
        print(f"  Total de valores nulos: {total_nulls:,}")
        print()
        
        # Guardar snapshot para comparação
        if return_snapshots:
            snapshots[table_name] = {
                "total_rows": total_rows,
                "total_nulls": total_nulls,
                "columns_with_nulls": {c: null_counts_row[c] for c in cols if null_counts_row[c] > 0}
            }
    
    return snapshots if return_snapshots else None




In [0]:
def comparar_nulos(snapshot_before, snapshot_after, table_list):
    """
    Compara snapshots de nulos antes e depois de uma transformação.
    
    Parâmetros:
    -----------
    snapshot_before : dict
        Snapshot de nulos antes da transformação (retornado por analisar_nulos)
    snapshot_after : dict
        Snapshot de nulos depois da transformação (retornado por analisar_nulos)
    table_list : list
        Lista de nomes de tabelas a comparar
    
    Retorna:
    --------
    bool : True se houve perda de dados detectada, False caso contrário
    """
    
    print("=" * 70)
    print("COMPARAÇÃO: DADOS PERDIDOS DURANTE A TRANSFORMAÇÃO?")
    print("=" * 70 + "\n")
    
    data_loss_detected = False
    
    for table_name in table_list:
        before = snapshot_before[table_name]
        after = snapshot_after[table_name]
        
        nulls_gained = after["total_nulls"] - before["total_nulls"]
        
        print(f"Tabela {table_name.upper()}:")
        
        if nulls_gained > 0:
            print(f"  ⚠️  ATENÇÃO: Aumentaram {nulls_gained:,} valores nulos!")
            print(f"     Antes: {before['total_nulls']:,} nulos | Depois: {after['total_nulls']:,} nulos")
            data_loss_detected = True
            
            # Identificar quais colunas ganharam nulos
            print(f"\n  Colunas afetadas:")
            for col in after["columns_with_nulls"]:
                nulls_before = before["columns_with_nulls"].get(col, 0)
                nulls_after = after["columns_with_nulls"][col]
                
                if nulls_after > nulls_before:
                    print(f"    - {col}: {nulls_before:,} → {nulls_after:,} (+{nulls_after - nulls_before:,})")
        
        elif nulls_gained < 0:
            print(f"  ✅ Diminuíram {abs(nulls_gained):,} valores nulos (inesperado, mas positivo)")
        
        else:
            print(f"  ✅ Nenhuma perda de dados detectada")
            print(f"     Antes: {before['total_nulls']:,} nulos | Depois: {after['total_nulls']:,} nulos")
        
        print()
    
    if data_loss_detected:
        print("\n⚠️  CONCLUSÃO: Foram detetados valores que se tornaram nulos.")
        print("   Isto pode indicar conversões falhadas (ex: strings inválidas em colunas de data/número).")
        print("   Recomenda-se investigar as colunas afetadas antes de prosseguir.\n")
    else:
        print("\n✅ CONCLUSÃO: Não houve perda de dados durante a transformação.")
        print("   Todas as conversões de tipo foram bem-sucedidas.\n")
    
    return data_loss_detected

print("✅ Funções analisar_nulos() e comparar_nulos() carregadas com sucesso!")

In [0]:
# Primeiro carregamos as tabelas Delta da camada Bronze
bronze_dfs = {}
for table in tables:
    bronze_dfs[table] = spark.read.format("delta").load(f"{bronze_delta_path}/{table}/")


In [0]:
# visualização das tabelas
for table in tables:
    print(f"\nTabela: {table}")
    display(bronze_dfs[table].limit(5))


### 3.2 Schema Casting (Conversão de Tipos de Dados)

Na camada Bronze, todas as colunas foram ingeridas temporariamente como `string` para garantir a fidelidade aos ficheiros originais. Agora, na camada Silver, é necessário atribuir os tipos de dados semânticos corretos (Datas, Números Inteiros e Decimais).

**Principais Transformações:**
1. **Datas:** Os ficheiros FAERS utilizam o formato `AAAAMMDD`. Colunas como `event_dt` ou `fda_dt` serão convertidas para `DateType`.*
2. **Métricas Clínicas e Doses:** Colunas quantitativas como `age` (idade), `wt` (peso) e `dose_amt` (quantidade da dose) serão convertidas para `IntergerType` e `DoubleType` respectivamente, para permitir agregações matemáticas (médias, distribuições) na camada Gold.
3. As variáveis categóricas e identificadores (como `primaryid`, `pt`, `outc_cod`) mantêm-se como `string`.


In [0]:
# Contagem de nulos ANTES do casting (Bronze)
print("=== CONTAGEM DE NULOS - ANTES DO SCHEMA CASTING ===")
print("(Dados ainda em formato string na camada Bronze)\n")

null_counts_before = analisar_nulos(
    dfs_dict=bronze_dfs,
    return_snapshots=True
)

print("✅ Snapshot guardado em 'null_counts_before'\n")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

silver_dfs = {}
print("=== SCHEMA CASTING E GRAVAÇÃO SILVER ===\n")

for table_name, df in bronze_dfs.items():
    df_cast = df
    
    # Transformações específicas para a tabela DEMO
    if table_name == "demo":
        df_cast = df_cast.withColumn("event_dt", F.to_date(F.col("event_dt"), "yyyyMMdd")) \
                         .withColumn("mfr_dt", F.to_date(F.col("mfr_dt"), "yyyyMMdd")) \
                         .withColumn("init_fda_dt", F.to_date(F.col("init_fda_dt"), "yyyyMMdd")) \
                         .withColumn("fda_dt", F.to_date(F.col("fda_dt"), "yyyyMMdd")) \
                         .withColumn("rept_dt", F.to_date(F.col("rept_dt"), "yyyyMMdd")) \
                         .withColumn("age", F.col("age").cast(IntegerType())) \
                         .withColumn("wt", F.col("wt").cast(DoubleType()))
                         
    # Transformações específicas para a tabela DRUG
    elif table_name == "drug":
        df_cast = df_cast.withColumn("exp_dt", F.to_date(F.col("exp_dt"), "yyyyMMdd")) \
                         .withColumn("dose_amt", F.col("dose_amt").cast(DoubleType())) \
                         .withColumn("cum_dose_chr", F.col("cum_dose_chr").cast(DoubleType()))
                         
    # As tabelas REAC e OUTC contêm apenas identificadores e códigos em texto, 
    # pelo que não necessitam de casting numérico/temporal.
    
    silver_dfs[table_name] = df_cast
    print(f"Schema atualizado para a tabela: {table_name.upper()}")

print("\n--- A Iniciar Gravação na Camada Silver ---")

# Gravação em formato Delta
for table_name, df in silver_dfs.items():
    output_path = f"{silver_delta_path}/{table_name}"
    
    (
        df.write
          .mode("overwrite")
          .format("delta")
          .option("overwriteSchema", "true")
          .save(output_path)
    )
    print(f"✅ Tabela {table_name.upper()} gravada com sucesso em: {output_path}")




In [0]:
# Contagem de nulos DEPOIS do casting (Silver) e comparação
print("\n=== CONTAGEM DE NULOS - DEPOIS DO SCHEMA CASTING ===")
print("(Dados convertidos para tipos semânticos na camada Silver)\n")

null_counts_after = analisar_nulos(
    dfs_dict=silver_dfs,
    return_snapshots=True
)

print("✅ Snapshot guardado em 'null_counts_after'\n")

# Comparar antes vs depois
data_loss = comparar_nulos(
    snapshot_before=null_counts_before,
    snapshot_after=null_counts_after,
    table_list=tables
)


### 3.3 Profiling inicial
Após verificarmos que as tabelas bronze foram corretamente carregadas e que não houve perda de dados ao aplicar o schema pretendido, vamos começar a fazer um profiling inicial.

Nesta fase queremos perceber a estrutura e qualidade dos dados antes de definir regras de limpeza.


### 3.3.1 Contagens e schemas

In [0]:
# Faz-se uma contagem do nº de registos e de colunas de cada tabela na fase bronze.
# Neste caso, todas as tabelas foram carregadas com um schema que define todas as colunas como string.
# Ainda assim, é importante realizar uma última verificação.
for table, df in silver_dfs.items():
    print(f"Tabela {table.upper()} apresenta {bronze_dfs[table].count():,} registos.")
    print(f"Tabela {table.upper()} apresenta {len(bronze_dfs[table].columns)} colunas.")
    print(f"\nSchema da tabela {table.upper()}:")
    df.printSchema()


### 3.2.2 Verificação de nulos
Após a leitura das tabelas e a análise dos respetivos schemas, procede-se à verificação de valores nulos em cada coluna.

O objetivo desta etapa é avaliar a completude dos dados antes da aplicação das regras de limpeza da camada Silver. Para cada tabela, será calculado o número de valores nulos por coluna e a respetiva percentagem face ao total de registos.

Esta análise permite distinguir entre nulos em campos críticos, como `primaryid` e `caseid`, que podem comprometer a integridade relacional dos dados, e nulos em campos opcionais ou clinicamente informativos, cuja ausência pode ser esperada e deve ser preservada ou tratada com cautela.


In [0]:
# Reutilizar a função analisar_nulos() para profiling
print("=== ANÁLISE DETALHADA DE NULOS (Profiling) ===\n")

# Usar função para totais
_ = analisar_nulos(
    dfs_dict=silver_dfs,
    return_snapshots=False
)

# Análise detalhada por coluna
print("\n=== ANÁLISE DETALHADA POR COLUNA ===\n")

for table_name, df in silver_dfs.items():
    print(f"--- Tabela: {table_name.upper()} - Nulos por Coluna ---")
    
    cols = df.columns
    total_rows = df.count()
    
    # Contagem eficiente: uma única passagem
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in cols]
    null_counts_row = df.select(*null_exprs).collect()[0]
    
    # Construir lista para display
    null_data = []
    for col_name in cols:
        null_count = null_counts_row[col_name]
        null_pct = round((null_count / total_rows) * 100, 1) if total_rows > 0 else 0
        null_data.append((col_name, null_count, null_pct))
    
    # Criar DataFrame para display ordenado
    null_df = spark.createDataFrame(
        null_data,
        ["column_name", "null_count", "null_pct"]
    ).orderBy(F.desc("null_pct"))
    
    display(null_df)
    print()

### 3.2.3 Verificação de duplicados

A análise de duplicados será usada para definir a estratégia de limpeza na camada Silver.

Na camada Silver serão removidos duplicados exatos e serão mantidos os identificadores necessários para preservar relações entre tabelas. *A deduplicação por chave lógica será aplicada com cuidado, uma vez que algumas tabelas FAERS podem conter múltiplos registos válidos por caso, medicamento, reação ou desfecho.*


In [0]:
print("=== VERIFICAÇÃO DE DUPLICADOS EXATOS ===\n")

for table, df in bronze_dfs.items():
    print(f"--- Tabela: {table.upper()} ---")
    
    total_rows = df.count()
    distinct_rows = df.distinct().count()
    duplicate_rows = total_rows - distinct_rows
    
    print(f"  Total de registos: {total_rows:,}")
    print(f"  Registos distintos: {distinct_rows:,}")
    print(f"  Duplicados exatos: {duplicate_rows:,}")
    print()

### 3.2.3 Duplicados por chave lógica

Ao contrário dos duplicados exatos, onde todas as colunas da linha são iguais, os duplicados por chave lógica ocorrem quando existem várias linhas com a mesma combinação de campos identificadores. Estes casos podem representar duplicação real, mas também podem refletir a estrutura natural dos dados FAERS.

Nesta análise, não serão removidos registos automaticamente. O objetivo é apenas identificar situações em que a mesma chave lógica aparece mais do que uma vez, para compreender se esses casos exigem tratamento posterior na camada Silver.

As chaves lógicas consideradas são:

| Tabela | Chave lógica utilizada | Interpretação |
|---|---|---|
| `demo` | `primaryid`, `caseid`, `caseversion` | identifica uma versão específica de um caso |
| `drug` | `primaryid`, `caseid`, `drug_seq` | identifica um medicamento dentro de um caso |
| `reac` | `primaryid`, `caseid`, `pt` | identifica uma reação reportada num caso |
| `outc` | `primaryid`, `caseid`, `outc_cod` | identifica um outcome associado a um caso |

Esta análise permite perceber se existem combinações de chaves repetidas e avaliar se devem ser mantidas, investigadas ou removidas numa fase posterior. Em tabelas como `drug`, `reac` e `outc`, a existência de várias linhas por caso pode ser esperada, uma vez que um mesmo caso pode estar associado a múltiplos medicamentos, reações ou outcomes.

In [0]:
# definir função para identificar duplicados por chave lógica
def analisar_duplicados_chave_logica(df, table_name, key_cols, show_results=True):
    """
    Analisa duplicados por chave lógica numa tabela.

    Parâmetros:
    df : DataFrame
        DataFrame a analisar.
    table_name : str
        Nome da tabela, usado apenas para identificação no output.
    key_cols : list
        Lista de colunas que compõem a chave lógica.
    show_results : bool
        Se True, mostra o resultado com display().

    Retorna:
    DataFrame com as combinações de chave lógica que aparecem mais do que uma vez.
    """

    print(f"--- Duplicados por chave lógica: {table_name.upper()} ---")

    # Confirma se todas as colunas da chave existem na tabela
    missing_cols = [c for c in key_cols if c not in df.columns]

    if missing_cols:
        print(f"Colunas em falta para esta análise: {missing_cols}\n")
        return None

    # Agrupa pela chave lógica e identifica combinações repetidas
    duplicate_keys_df = (
        df.groupBy(key_cols)
          .count()
          .filter(F.col("count") > 1)
          .orderBy(F.desc("count"))
    )

    total_duplicate_keys = duplicate_keys_df.count()

    print(f"Número de chaves lógicas com mais de um registo: {total_duplicate_keys}")

    if show_results:
        display(duplicate_keys_df.limit(5))

    return duplicate_keys_df

In [0]:
# Define um dicionário com as chaves lógicas a usar em cada tabela.
logical_keys = {
    "demo": ["primaryid", "caseid", "caseversion"],
    "drug": ["primaryid", "caseid", "drug_seq"],
    "reac": ["primaryid", "caseid", "pt"],
    "outc": ["primaryid", "caseid", "outc_cod"]
}

duplicate_keys_dfs = {}

# analise duplicados por chave lógica em cada tabela.
for table_name, key_cols in logical_keys.items():
    df = silver_dfs[table_name]

    duplicate_keys_dfs[table_name] = analisar_duplicados_chave_logica(
        df=df,
        table_name=table_name,
        key_cols=key_cols,
        show_results=True
    )

In [0]:
# Verificar se os duplicados por chave lógica são registos exatos ou únicos
import pyspark.sql.functions as F

print("=== INSPEÇÃO DE DUPLICADOS POR CHAVE LÓGICA ===\n")
print("Objetivo: Verificar se registos com a mesma chave lógica são idênticos (duplicados exatos)")
print("ou se são registos únicos que partilham apenas a chave.\n")

for table_name, key_cols in logical_keys.items():
    print(f"\n{'='*70}")
    print(f"Tabela: {table_name.upper()}")
    print(f"Chave lógica: {', '.join(key_cols)}")
    print(f"{'='*70}\n")
    
    df = silver_dfs[table_name]
    duplicate_keys_df = duplicate_keys_dfs[table_name]
    
    if duplicate_keys_df is None or duplicate_keys_df.count() == 0:
        print("✅ Não há duplicados por chave lógica nesta tabela.\n")
        continue
    
    # Pegar a primeira chave duplicada como exemplo
    first_duplicate_key = duplicate_keys_df.first()
    
    # Construir filtro para essa chave
    filter_condition = None
    for col in key_cols:
        if filter_condition is None:
            filter_condition = (F.col(col) == first_duplicate_key[col])
        else:
            filter_condition = filter_condition & (F.col(col) == first_duplicate_key[col])
    
    # Buscar todos os registos com essa chave
    duplicate_records = df.filter(filter_condition)
    num_duplicates = duplicate_records.count()
    
    print(f"Exemplo de chave duplicada:")
    for col in key_cols:
        print(f"  {col}: {first_duplicate_key[col]}")
    print(f"\nNúmero de registos com esta chave: {num_duplicates}")
    
    # Verificar se são duplicados exatos
    distinct_records = duplicate_records.distinct().count()
    
    if distinct_records == 1:
        print(f"\n⚠️  DUPLICADOS EXATOS: Todos os {num_duplicates} registos são idênticos.")
        print("   Ação recomendada: Remover duplicados com dropDuplicates().")
    else:
        print(f"\n✅ REGISTOS ÚNICOS: {distinct_records} registos distintos partilham a mesma chave.")
        print("   Ação recomendada: Manter registos (são dados válidos).")
    
    print(f"\nAmostra dos primeiros 3 registos:")
    display(duplicate_records.limit(3))
    print()

### 3.2.4 Verificação de ID

Nesta etapa verificamos se a coluna de identificação do dataset contém valores únicos para cada registo.

A unicidade do ID é importante porque em muitos datasets o identificador deve representar uma entidade ou observação única. Se existirem IDs repetidos, isso pode indicar duplicação de registos, erros de integração, problemas no carregamento dos dados ou situações específicas do domínio que precisam de ser analisadas.

A tabela `DEMO` contem o primaryID de todos os casos que constam em todas as tabelas, sendo que esta chave é concatenada do ID do Caso e do Número da Versão do Caso.

É possivel e aceitável que `primaryid` tenha duplicados noutras tabelas pois cada caso pode ter varias drogas/reações/outcomes associados


In [0]:
# Verificar se primaryid é único na tabela DEMO
import pyspark.sql.functions as F

print("=== VERIFICAÇÃO DE UNICIDADE DO primaryid NA DEMO ===\n")
print("primaryid deve ser a chave primária única só na tabela DEMO.\n")

df_demo = silver_dfs["demo"]

total_rows = df_demo.count()
distinct_primaryid = df_demo.select("primaryid").distinct().count()
duplicate_primaryid = total_rows - distinct_primaryid

print(f"  Total de registos: {total_rows:,}")
print(f"  primaryid distintos: {distinct_primaryid:,}")

if duplicate_primaryid > 0:
    print(f"  ⚠️  ATENÇÃO: {duplicate_primaryid:,} registos com primaryid duplicado!")
    print(f"     primaryid NÃO é único na tabela DEMO.")
else:
    print(f"  ✅ primaryid é único (sem duplicados)")

print()

### 3.2.5 Verificação de integridade de referência / deteção de entradas orfãs

Entradas órfãs nas tabelas filhas, isto é, registos com `primaryid` sem correspondência na tabela `DEMO`, não conseguem fazer join com a tabela principal na camada Gold.

Isto pode levar à perda desses registos nas análises finais ou à criação de resultados incompletos. 

Por isso, nesta fase verificamos se todos os `primaryid` presentes nas tabelas filhas existem também na `DEMO`.

In [0]:
# Verificar se todos os primaryid nas tabelas filhas existem na tabela DEMO
import pyspark.sql.functions as F

print("=== VERIFICAÇÃO DE INTEGRIDADE REFERENCIAL ===\n")
print("Verificar se todos os primaryid em DRUG, REAC e OUTC existem em DEMO.\n")

df_demo = silver_dfs["demo"]
valid_primaryids = df_demo.select("primaryid").distinct()

child_tables = ["drug", "reac", "outc"]

for table_name in child_tables:
    print(f"--- Tabela: {table_name.upper()} ---")
    
    df_child = silver_dfs[table_name]
    
    # Contar primaryids únicos na tabela filha
    total_primaryids = df_child.select("primaryid").distinct().count()
    
    # Encontrar primaryids órfãos (não existem em DEMO)
    orphan_primaryids = (
        df_child.select("primaryid")
        .distinct()
        .join(valid_primaryids, on="primaryid", how="left_anti")
    )
    
    orphan_count = orphan_primaryids.count()
    
    # Contar registos órfãos (não apenas IDs únicos)
    orphan_records = df_child.join(orphan_primaryids, on="primaryid", how="inner")
    orphan_records_count = orphan_records.count()
    
    print(f"  primaryid únicos na tabela: {total_primaryids:,}")
    print(f"  primaryid órfãos (não existem em DEMO): {orphan_count:,}")
    print(f"  Registos órfãos: {orphan_records_count:,}")
    
    if orphan_count > 0:
        print(f"  ⚠️  ATENÇÃO: Existem registos órfãos!")
        print(f"     Ação recomendada: Remover ou investigar estes registos.")
    else:
        print(f"  ✅ Integridade referencial OK (todos os primaryid existem em DEMO)")
    
    print()

### 3.2.4 Valores distintos em variáveis categóricas

Nesta etapa serão analisados os valores distintos existentes em algumas colunas categóricas relevantes das tabelas FAERS.

O objetivo é perceber que códigos e categorias aparecem nos dados antes da aplicação das regras de limpeza da camada Silver. Esta análise permite identificar inconsistências como diferenças de capitalização, espaços em branco, valores pouco frequentes, categorias desconhecidas ou códigos que possam necessitar de normalização.

Serão analisadas sobretudo colunas com significado categórico, como sexo, país do reporter, tipo de reporter, papel do medicamento no caso, via de administração, resultados clínicos e códigos de desfecho.

Para cada coluna selecionada, será apresentada a contagem de ocorrências por valor distinto, ordenada de forma decrescente. Esta informação será usada posteriormente para justificar transformações como `trim`, `upper` e o preenchimento de valores desconhecidos com códigos como `UNK` ou `U`.

In [0]:
import pyspark.sql.functions as F

categorical_cols = {
    "demo": ["sex", "occp_cod", "reporter_country", "e_sub"],
    "drug": ["role_cod", "route", "dechal", "rechal", "dose_freq"],
    "reac": ["pt"],
    "outc": ["outc_cod"]
}

for table_name, cols in categorical_cols.items():
    print(f"--- Valores distintos em variáveis categóricas: {table_name.upper()} ---")
    
    df = silver_dfs[table_name]
    
    for col_name in cols:
        if col_name not in df.columns:
            print(f"A coluna '{col_name}' não existe na tabela {table_name}.\n")
            continue
        
        print(f"Coluna: {col_name}")
        
        distinct_values_df = (
            df.groupBy(col_name)
              .count()
              .orderBy(F.desc("count"))
              .limit(30)
        )
        
        display(distinct_values_df)

### 3.3.5 Análise de Variáveis Numéricas

Após a análise de variáveis categóricas, é importante explorar as variáveis quantitativas para identificar:
- **Distribuições**: valores mínimos, máximos, médias e medianas.
- **Outliers**: valores extremos ou clinicamente implausíveis (ex: idades negativas, pesos acima de 500 kg).
- **Missing patterns**: verificar se existem padrões de ausência em doses ou métricas clínicas.

As principais variáveis numéricas nas tabelas FAERS são:

| Tabela | Variável | Descrição |
|--------|----------|------------|
| `demo` | `age` | Idade do paciente |
| `demo` | `wt` | Peso do paciente (kg) |
| `drug` | `dose_amt` | Quantidade da dose administrada |
| `drug` | `cum_dose_chr` | Dose cumulativa |

Esta análise ajudará a definir regras de limpeza para valores extremos e a decidir estratégias de imputação ou filtragem na camada Silver.

In [0]:
import pyspark.sql.functions as F

print("=== DISTRIBUIÇÃO DE CÓDIGOS DE UNIDADE ===\n")

# Analisar distribuição de age_cod na tabela demo
print("--- Tabela: DEMO | Coluna: age_cod ---")
df_demo = silver_dfs["demo"]

age_cod_dist = (
    df_demo.groupBy("age_cod")
    .agg(
        F.count("*").alias("count"),
        F.round((F.count("*") / df_demo.count()) * 100, 2).alias("percentage")
    )
    .orderBy(F.desc("count"))
)

display(age_cod_dist)

print("\nSignificado dos códigos de age_cod:")
print("  YR  = Anos (Years)")
print("  DY  = Dias (Days)")
print("  DEC = Décadas (Decades)")
print("  MON = Meses (Months)")
print("  WK  = Semanas (Weeks)")
print("  HR  = Horas (Hours)")

print("\n" + "-"*60 + "\n")

# Analisar distribuição de wt_cod na tabela demo
print("--- Tabela: DEMO | Coluna: wt_cod ---")

wt_cod_dist = (
    df_demo.groupBy("wt_cod")
    .agg(
        F.count("*").alias("count"),
        F.round((F.count("*") / df_demo.count()) * 100, 2).alias("percentage")
    )
    .orderBy(F.desc("count"))
)

display(wt_cod_dist)

print("\nSignificado dos códigos de wt_cod:")
print("  KG  = Quilogramas (Kilograms)")
print("  LBS = Libras (Pounds)")
print("  GMS = Gramas (Grams)")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType, IntegerType

# Definir colunas numéricas por tabela
numerical_cols = {
    "demo": ["age", "wt"],
    "drug": ["dose_amt", "cum_dose_chr"]
}

for table_name, cols in numerical_cols.items():
    print(f"\n{'='*60}")
    print(f"Análise de Variáveis Numéricas: {table_name.upper()}")
    print(f"{'='*60}\n")
    
    df = silver_dfs[table_name]
    
    for col_name in cols:
        if col_name not in df.columns:
            print(f"⚠️  Coluna '{col_name}' não existe na tabela {table_name}.\n")
            continue
        
        print(f"\n--- Coluna: {col_name.upper()} ---")
        
        # Para a coluna 'age', filtrar apenas registos onde age_cod = 'YR'
        # Para a coluna 'wt', filtrar apenas registos onde wt_cod = 'KG'
        if col_name == "age" and "age_cod" in df.columns:
            print("⚠️  Filtro aplicado: age_cod = 'YR' (apenas idades em anos)")
            df_filtered = df.filter(F.col("age_cod") == "YR")
        elif col_name == "wt" and "wt_cod" in df.columns:
            print("⚠️  Filtro aplicado: wt_cod = 'KG' (apenas pesos em quilogramas)")
            df_filtered = df.filter(F.col("wt_cod") == "KG")
        else:
            df_filtered = df
        
        # Estatísticas descritivas usando describe()
        stats_df = df_filtered.select(col_name).describe()
        display(stats_df)
        
        # Análise adicional: valores negativos e zeros
        total_rows = df.count()
        total_rows_filtered = df_filtered.count()
        negative_count = df_filtered.filter(F.col(col_name) < 0).count()
        zero_count = df_filtered.filter(F.col(col_name) == 0).count()
        null_count = df_filtered.filter(F.col(col_name).isNull()).count()
        
        print(f"\n📊 Análise de Qualidade:")
        if col_name in ["age", "wt"]:
            print(f"   Total de registos (tabela completa): {total_rows:,}")
            print(f"   Total de registos após filtro: {total_rows_filtered:,}")
        else:
            print(f"   Total de registos: {total_rows:,}")
        print(f"   Valores nulos: {null_count:,} ({round(null_count/total_rows_filtered*100, 2) if total_rows_filtered > 0 else 0}%)")
        print(f"   Valores negativos: {negative_count:,} ({round(negative_count/total_rows_filtered*100, 2) if total_rows_filtered > 0 else 0}%)")
        print(f"   Valores zero: {zero_count:,} ({round(zero_count/total_rows_filtered*100, 2) if total_rows_filtered > 0 else 0}%)")
        print(f"   Valores válidos (não-nulos e positivos): {total_rows_filtered - null_count - negative_count:,}")
        print("-" * 60)

In [0]:
import pyspark.sql.functions as F

print("\n" + "="*80)
print("ANÁLISE DE OUTLIERS - PERCENTIS E VALORES EXTREMOS")
print("="*80 + "\n")

# Analisar percentis para detectar outliers
numerical_cols = {
    "demo": ["age", "wt"],
    "drug": ["dose_amt", "cum_dose_chr"]
}

for table_name, cols in numerical_cols.items():
    print(f"\n--- Tabela: {table_name.upper()} ---\n")
    
    df = silver_dfs[table_name]
    
    for col_name in cols:
        if col_name not in df.columns:
            continue
        
        print(f"Coluna: {col_name.upper()}")
        
        # Para a coluna 'age', filtrar apenas registos onde age_cod = 'YR'
        # Para a coluna 'wt', filtrar apenas registos onde wt_cod = 'KG'
        if col_name == "age" and "age_cod" in df.columns:
            print("⚠️  Filtro aplicado: age_cod = 'YR' (apenas idades em anos)\n")
            df_filtered = df.filter(F.col("age_cod") == "YR")
        elif col_name == "wt" and "wt_cod" in df.columns:
            print("⚠️  Filtro aplicado: wt_cod = 'KG' (apenas pesos em quilogramas)\n")
            df_filtered = df.filter(F.col("wt_cod") == "KG")
        else:
            df_filtered = df
        
        # Calcular percentis (1%, 5%, 25%, 50%, 75%, 95%, 99%)
        percentiles = df_filtered.stat.approxQuantile(
            col_name, 
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99], 
            0.01
        )
        
        print(f"   P1:  {percentiles[0]}")
        print(f"   P5:  {percentiles[1]}")
        print(f"   P25: {percentiles[2]}")
        print(f"   P50 (Mediana): {percentiles[3]}")
        print(f"   P75: {percentiles[4]}")
        print(f"   P95: {percentiles[5]}")
        print(f"   P99: {percentiles[6]}")
        
        # Mostrar os 10 valores mais extremos (máximos)
        print(f"\n   Top 10 valores mais altos:")
        top_values = (
            df_filtered.select(col_name)
              .filter(F.col(col_name).isNotNull())
              .orderBy(F.desc(col_name))
              .limit(10)
        )
        display(top_values)
        
        print("-" * 60)

## 3.4 Síntese dos Resultados do Profiling

Após análise exaustiva das 4 tabelas FAERS (DEMO, DRUG, REAC, OUTC), foram identificados os seguintes padrões e problemas de qualidade de dados:

---

### 📊 **Estrutura e Volume**
* **DEMO**: 2,157,280 registos (casos únicos)
* **DRUG**: 9,480,689 registos 
* **REAC**: 7,461,401 registos
* **OUTC**: 1,582,534 registos
* **Schema Casting**: Colunas de data convertidas de string (`AAAAMMDD`) para `DateType` e variáveis numéricas (`age`, `wt`, `dose_amt`) convertidas para Integer/Double **sem perda de dados**.

---

### ⚠️ **Valores Nulos (Missing Data)**

#### **Alta incidência de nulls em:**
* **age_cod** e **age**: 45.48% null (muitos casos sem informação de idade)
* **wt_cod**: 83.48% null (peso raramente reportado)
* **event_dt**, **mfr_dt**: Percentagem elevada de nulls em datas
* Campos opcionais de dosagem (`dose_amt`, `cum_dose_chr`) também apresentam missingness significativo

**Implicação**: Análises envolvendo idade ou peso terão amostras reduzidas. Será necessário decidir entre **imputação**, **filtragem** ou **criação de categoria "Desconhecido"** na camada Silver.

---

### 🔁 **Duplicados Identificados**

Foram realizadas duas verificações distintas:

#### **1. Duplicados Exatos (todas as colunas idênticas)**

| Tabela | Total de Registos | Duplicados Exatos Encontrados | % Duplicados |
|--------|-------------------|-------------------------------|-------------|
| DEMO   | 2,157,280         | **0**                         | 0%          |
| DRUG   | 9,480,689         | **4**                         | 0%          |
| REAC   | 7,461,401         | **108,888**                   | **1.46%**   |
| OUTC   | 1,582,534         | **0**                         | 0%          |

**Observação crítica**: A tabela `REAC` contém mais de 100 mil duplicados exatos, provavelmente devido a resubmissões de follow-ups ou erros de preenchimento.

#### **2. Duplicados por Chave Lógica (Business Key)**

Além da verificação de duplicados exatos, foi analisada a unicidade das chaves lógicas em cada tabela:

* **DEMO**: Chave lógica = `primaryid` → **0 duplicados** (validado)
* **DRUG**: Chave lógica = `(primaryid, drug_seq)` → análise realizada
* **REAC**: Chave lógica = `(primaryid, pt)` → análise realizada  
* **OUTC**: Chave lógica = `(primaryid, outc_cod)` → análise realizada

Esta análise permitiu identificar casos onde o mesmo paciente reporta o mesmo medicamento ou reação múltiplas vezes, que podem ser legítimos (doses repetidas) ou redundantes.

**Ação recomendada**: Remover duplicados exatos para evitar *double-counting* nas análises.

---

### 🔑 **Integridade de Chaves**

#### ✅ **Primary Key (DEMO)**
* `primaryid` é **único** na tabela DEMO (2,157,280 valores distintos)
* Sem duplicados → validação bem-sucedida

#### ✅ **Integridade Referencial**
* **0 registos órfãos** em todas as tabelas filhas (DRUG, REAC, OUTC)
* Todos os `primaryid` nas tabelas filhas existem na DEMO
* Relação 1:N garantida para joins na camada Gold

---

### 🏷️ **Códigos de Unidade (Unit Codes)**

#### **age_cod** (unidade de idade)
* **52.99%** em Anos (`YR`) — **maioria utilizável diretamente**
* 45.48% null
* Restantes em DEC, DY, MON, WK, HR (< 2% combinados)

**Ação recomendada**: Normalizar todas as unidades para **anos** (converter DEC, MON, DY, etc.) antes de análises de idade.

#### **wt_cod** (unidade de peso)
* **83.48%** null
* **16.32%** em Quilogramas (`KG`)
* 0.21% em Libras (`LBS`)

**Ação recomendada**: Converter LBS → KG e marcar nulls como "Peso não reportado".

---

### 📈 **Variáveis Categóricas — Dispersão**

Foram analisadas as principais variáveis categóricas para identificar inconsistências:

* **sex** (DEMO): Valores esperados (M, F) mas podem existir nulls, "UNK", ou códigos inconsistentes
* **reporter_country** (DEMO): Grande dispersão geográfica — necessário validar códigos ISO
* **role_cod** (DRUG): Códigos como `PS` (Primary Suspect), `SS` (Secondary Suspect), `C` (Concomitant)
* **route** (DRUG): Via de administração — alta cardinalidade (oral, intravenoso, etc.)
* **pt** (REAC): Preferred Terms (MedDRA) — milhares de valores únicos
* **outc_cod** (OUTC): Códigos de desfecho clínico (`DE` = Death, `HO` = Hospitalization, etc.)

**Observação**: Não foram identificadas inconsistências críticas de capitalização ou espaços em branco, mas será aplicada normalização (`trim()`, `upper()`) por precaução.

---

### 🚨 **Outliers em Variáveis Numéricas**

#### **Age (em anos, filtro age_cod = 'YR')**
* **Mediana**: 60 anos
* **P95**: 84 anos (plausível)
* **P99**: 987 anos ⚠️ **OUTLIER EXTREMO**
* **Máximo**: 987 anos

**Diagnóstico**: Idades acima de 120 anos são **clinicamente impossíveis**. Valores como 987 são provavelmente erros de digitação ou códigos placeholder.

**Ação recomendada**: Filtrar registos com `age > 120` ou marcar como inválidos.

#### **Weight (em KG, filtro wt_cod = 'KG')**
* **Mediana**: ~70 kg (razoável)
* **P99**: ~200 kg (obeso mórbido, mas plausível)
* **Máximo**: 87,075 kg ⚠️ **OUTLIER EXTREMO**

**Diagnóstico**: Pesos acima de 500 kg são **impossíveis** para humanos. Valores como 87,075 são erros de entrada ou conversão incorreta de unidades.

**Ação recomendada**: Filtrar registos com `wt > 500` ou investigar conversão de unidades.

#### **dose_amt, cum_dose_chr (DRUG)**
* Alta variabilidade devido a diferentes medicamentos e unidades de dosagem
* Valores extremos identificados — necessário análise contextual por medicamento

---

### 🎯 **Próximos Passos (Data Cleaning)**

Com base nos achados do profiling, as seguintes transformações serão aplicadas na camada Silver:

1. **Remoção de duplicados exatos**:
   * Remover 108,888 duplicados da tabela REAC
   * Remover 4 duplicados da tabela DRUG
2. **Normalização de unidades**:
   * Converter todas as idades para anos
   * Converter pesos LBS → KG
3. **Tratamento de outliers**:
   * Filtrar `age > 120`
   * Filtrar `wt > 500`
4. **Normalização de strings categóricas**:
   * Aplicar `trim()` e `upper()` em códigos categóricos
   * Substituir nulls por `'UNK'` ou `'U'` onde apropriado
5. **Validação de datas**:
   * Remover datas no futuro
   * Verificar ordem cronológica (event_dt < fda_dt)
6. **Criação de tabela analítica**:
   * Join de DEMO + DRUG + REAC + OUTC para análises da camada Gold

## 3.5 Data Cleaning (Transformações da Camada Silver)

Com base nos achados do profiling, aplicaremos agora as transformações necessárias para criar a camada Silver limpa e pronta para análise.

As transformações seguirão a mesma estrutura do profiling:

1. **Tratamento de Valores Nulos** — preencher ou remover nulls conforme estratégia definida
2. **Remoção de Duplicados Exatos** — eliminar registos redundantes identificados
3. **Normalização de Unidades** — converter idades e pesos para unidades consistentes
4. **Remoção de Outliers** — filtrar valores clinicamente impossíveis
5. **Normalização de Strings Categóricas** — aplicar trim() e upper() em códigos
6. **Validação de Datas** — remover datas no futuro e validar ordem cronológica

Cada transformação será aplicada aos DataFrames no dicionário `silver_dfs` e os resultados serão validados antes de prosseguir para a próxima etapa.

### 3.5.1 Tratamento de Valores Nulos

O profiling revelou elevadas taxas de valores nulos em várias colunas:
* **age/age_cod**: 45.48% null
* **wt/wt_cod**: 83.48% null
* **event_dt**, **mfr_dt**: Percentagem elevada de nulls

**Estratégia de Limpeza:**

Para variáveis categóricas onde o null representa "desconhecido" ou "não reportado", iremos preencher com códigos padronizados:
* **sex**: nulls preenchidos com `'UNK'` (unknown) — semanticamente equivalente ao `'UNK'` já existente (0.34% dos casos)
* **occp_cod**, **reporter_country**, **e_sub**: nulls preenchidos com `'UNK'`

Para variáveis numéricas como `age` e `wt`, os nulls serão **mantidos** (não iremos imputar valores fictícios). Análises que requerem idade ou peso deverão filtrar registos onde estes campos são não-nulos.

Esta abordagem garante que:
1. Não criamos dados artificiais (ex: age=0 faria parecer recém-nascidos)
2. Categorias "desconhecido" ficam explícitas e consistentes
3. Queries analíticas podem usar `WHERE age IS NOT NULL` para obter amostras válidas

**Casos Específicos:**

A tabela **REAC** contém a coluna `drug_rec_act`, que apresenta 99,9% de valores nulos. Por esse motivo, não será aplicada imputação ou substituição desses valores, uma vez que a coluna contém informação útil em apenas uma fração muito reduzida dos registos. Preencher estes valores com uma categoria artificial, como "UNK", poderia distorcer a distribuição dos dados sem acrescentar valor analítico relevante.

A tabela **OUTC** não apresenta valores nulos, pelo que não será aplicada qualquer operação de limpeza relacionada com valores em falta. Assim, evita-se a execução de transformações redundantes no pipeline de processamento em Spark.

In [0]:


print("=" * 80)
print("1. TABELA DEMO - Tratamento de Nulls com Smart Fill")
print("=" * 80 + "\n")

df_demo = silver_dfs["demo"]
df_demo_before_fillna = silver_dfs["demo"]

# Smart fill: validar e preencher colunas categóricas
df_demo_clean = (
    df_demo
    # 1. SEX: validar M/F, caso contrário "UNK" (inclui nulls e valores inválidos)
    .withColumn(
        "sex",
        F.when(F.col("sex").isin("M", "F"), F.col("sex"))
         .otherwise("UNK")
    )
    .withColumn(
        "occp_cod",
        F.coalesce(F.col("occp_cod"), F.lit("UNK"))
    )
    .withColumn(
        "reporter_country",
        F.coalesce(F.col("reporter_country"), F.lit("UNK"))
    )
    .withColumn(
        "e_sub",
        F.coalesce(F.col("e_sub"), F.lit("UNK"))
    )
    .withColumn(
        "occr_country",
        F.coalesce(F.col("occr_country"), F.lit("UNK"))
    )
    .withColumn(
        "age_cod",
        F.coalesce(F.col("age_cod"), F.lit("UNK"))
    )
    .withColumn(
        "wt_cod",
        F.coalesce(F.col("wt_cod"), F.lit("UNK"))
    )
)



print("\n➡️  DataFrame limpo guardado em: df_demo_clean")

In [0]:
# verificação se limpeza foi bem sucedida
print("📋 Amostra de registos após tratamento de nulls:\n")
display(
    df_demo_clean
    .select("primaryid", "sex", "age", "age_cod", "wt", "wt_cod")
    .limit(20)
)
print("💡 Verificar: colunas categóricas não devem ter NULLs, apenas 'UNK'")
print("\nAntes de limpeza de nulos:", df_demo_before.filter(F.col("sex").isNull()).count())
print("Depois de limpeza de nulos:", df_demo_clean.filter(F.col("sex").isNull()).count())
print("Antes de limpeza de nulos:", df_demo_before.filter(F.col("sex") == "UNK").count())
print("Depois de limpeza de nulos:", df_demo_clean.filter(F.col("sex") == "UNK").count())


In [0]:
print("=" * 80)
print("2. TABELA DRUG - Tratamento de Nulls")
print("=" * 80 + "\n")

df_drug = silver_dfs["drug"]

# Aplicar fillna
df_drug_clean = df_drug.fillna({
    "role_cod": "UNK",
    "route": "UNK",
    "dechal": "UNK",
    "rechal": "UNK"
})

# validação rápida
print(f"\n✓ role_cod nulls antes: {silver_dfs['drug'].filter(F.col('role_cod').isNull()).count():,}")
print(f"✓ role_cod nulls depois: {df_drug_clean.filter(F.col('role_cod').isNull()).count():,}")
print("\n➡️  DataFrame limpo guardado em: df_drug_clean")

In [0]:
print("=" * 80)
print("4. ATUALIZAR DICIONÁRIO SILVER_DFS")
print("=" * 80 + "\n")

print("📦 Atualizando dicionário silver_dfs com tabelas tratadas...\n")

silver_dfs["demo"] = df_demo_clean
silver_dfs["drug"] = df_drug_clean
# REAC e OUTC mantêm-se inalterados (já justificado no markdown)

print("✅ Tabelas atualizadas no dicionário silver_dfs!")
print("\n💡 RESUMO:")
print("   • DEMO: nulls preenchidos em sex, occp_cod, reporter_country, e_sub, occr_country, age_cod, wt_cod")
print("   • DRUG: nulls preenchidos em role_cod, route, dechal, rechal")
print("   • REAC: sem alterações (drug_rec_act tem 99.9% nulls)")
print("   • OUTC: sem alterações (zero nulls)")
print("   • Variáveis numéricas (age, wt, dose_amt): nulls MANTIDOS")

### 3.5.2 Remoção de Duplicados Exatos

O profiling revelou a presença de duplicados exatos, com particular incidência na tabela `REAC` (mais de 100 mil registos). No contexto do FAERS, isto ocorre frequentemente devido a redundâncias no preenchimento do formulário original ou em submissões de acompanhamento (*follow-ups*) onde os mesmos sintomas são recarregados.

**Estratégia de Limpeza:**
Uma vez que são duplicados exatos (todas as colunas contêm os mesmos valores), estes registos não acrescentam qualquer contexto clínico novo. Pelo contrário, mantê-los causaria enviesamento e dupla contagem (*double-counting*) na fase de modelação ou na criação de dashboards. 

Aplica-se a função `dropDuplicates()` a todas as tabelas para garantir a integridade da camada Silver, mantendo apenas registos únicos para cada combinação de caso, medicamento e reação.

In [0]:
print("=== Tratamento De Duplicados Exatos===\n")

# Dicionário temporário para guardar os DataFrames sem duplicados
dedup_dfs = {}

for table_name, df in silver_dfs.items():
    # 1. Contagem inicial (antes da remoção)
    total_rows_antes = df.count()
    
    # 2. Remover duplicados exatos (avalia todas as colunas por defeito)
    df_dedup = df.dropDuplicates()
    
    # 3. Contagem final e cálculo da diferença
    total_rows_depois = df_dedup.count()
    duplicados_removidos = total_rows_antes - total_rows_depois
    
    # 4. Guardar o DataFrame limpo no novo dicionário
    dedup_dfs[table_name] = df_dedup
    
    # Mostrar resultados
    print(f"--- Tabela: {table_name.upper()} ---")
    print(f"Total antes: {total_rows_antes:,}")
    print(f"Duplicados removidos: {duplicados_removidos:,}")
    print(f"Total depois: {total_rows_depois:,}\n")

# Atualizar o dicionário principal com os dados agora sem nulos críticos e sem duplicados
silver_dfs = dedup_dfs


### 3.6 Validação Final da Camada Silver (Data Quality Check)

Antes de darmos a camada Silver como concluída, realizamos uma auditoria final diretamente nos ficheiros Delta que foram gravados no Unity Catalog/Volume. 

Esta validação garante que:
1. O motor Spark consegue ler as tabelas gravadas sem corrupção.
2. O **Schema Casting** foi persistido corretamente (verificando os tipos `date` e `double`).
3. Visualizamos uma amostra real dos dados já limpos de nulos críticos, sem duplicados e com a tipagem correta, prontos para alimentar a camada Gold.


In [0]:
print("=== AUDITORIA E VALIDAÇÃO DA CAMADA SILVER (DELTA) ===\n")

for table in tables:
    silver_path = f"{silver_delta_path}/{table}"
    
    print(f" Matriz de Validação para a tabela: {table.upper()}")
    
    # Ler diretamente do caminho Delta gravado
    df_silver = spark.read.format("delta").load(silver_path)
    
    # 1. Contagem total de linhas salvas
    total_rows = df_silver.count()
    print(f"   -> Total de registos persistidos: {total_rows:,}")
    print(f"   -> Total de colunas: {len(df_silver.columns)}")
    
    # 2. Print do Schema para validar visualmente o Casting
    print("   -> Estrutura do Schema:")
    df_silver.printSchema()
    
    # 3. Mostrar uma amostra rápida dos dados limpos
    print(f"   -> Amostra dos primeiros 3 registos de {table.upper()}:")
    display(df_silver.limit(3))
    
    print("-" * 80)
